In [0]:
# Day 3: PySpark Transformations Deep Dive
# Learn:
# - PySpark vs Pandas comparison
# - Joins (inner/left/right/full)
# - Window functions (running totals, rankings)
# - UDFs + derived features

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import StringType

In [0]:
# Load full dataset (Oct + Nov) and standardize

OCT_PATH = "/Volumes/workspace/ecommerce/ecommerce_data/2019-Oct.csv"
NOV_PATH = "/Volumes/workspace/ecommerce/ecommerce_data/2019-Nov.csv"

oct_raw = spark.read.option("header", True).option("inferSchema", True).csv(OCT_PATH)
nov_raw = spark.read.option("header", True).option("inferSchema", True).csv(NOV_PATH)

def standardize(df):
    return (
        df
        .withColumn("event_ts", F.to_timestamp(F.col("event_time")))
        .drop("event_time")
        .withColumn("price", F.col("price").cast("double"))
    )

events = standardize(oct_raw).unionByName(standardize(nov_raw))

print(f"Total events (Oct+Nov): {events.count():,}")
events.printSchema()

In [0]:
# PySpark vs Pandas (safe comparison)
spark_counts = (
    events.groupBy("event_type")
    .count()
    .orderBy(F.desc("count"))
)

spark_counts.show()

# Pandas (only for a small aggregate result)
pdf = spark_counts.limit(10).toPandas()
pdf

In [0]:
# Derived feature set (light Silver feature engineering)

events_feat = (
    events
    .withColumn("is_purchase", F.when(F.col("event_type") == "purchase", F.lit(1)).otherwise(F.lit(0)))
    .withColumn("revenue", F.when(F.col("event_type") == "purchase", F.col("price")).otherwise(F.lit(0.0)))
    .withColumn("event_date", F.to_date("event_ts"))
)

events_feat.select("event_ts","event_type","product_id","brand","price","is_purchase","revenue","event_date").show(5, truncate=False)


In [0]:
# Top 5 products by revenue (PDF practice adapted)
## No product_name, so we use product_id and optionally brand/category_code.

top_products_by_revenue = (
    events_feat
    .filter(F.col("event_type") == "purchase")
    .groupBy("product_id", "brand", "category_code")
    .agg(F.sum("revenue").alias("revenue"))
    .orderBy(F.desc("revenue"))
    .limit(5)
)

top_products_by_revenue.show(truncate=False)


In [0]:
# Joins (inner / left / right / full)

product_dim = (
    events_feat
    .select("product_id", "brand", "category_code")
    .dropDuplicates(["product_id"])
)

# INNER JOIN: only matching product_ids
inner_joined = top_products_by_revenue.join(product_dim, on="product_id", how="inner")

# LEFT JOIN: keep all top revenue products even if dim attributes are missing
left_joined = top_products_by_revenue.join(product_dim, on="product_id", how="left")

# RIGHT JOIN: keep all dim rows that match top_products (rarely useful here, but demo)
right_joined = top_products_by_revenue.join(product_dim, on="product_id", how="right")

# FULL OUTER JOIN: union of both sides
full_joined = top_products_by_revenue.join(product_dim, on="product_id", how="outer")

print("INNER:", inner_joined.count())
print("LEFT :", left_joined.count())
print("RIGHT:", right_joined.count())
print("FULL :", full_joined.count())

left_joined.show(5, truncate=False)


In [0]:
# Window function 1: Running event count per user (PDF practice, fixed)
w_user_running = (
    Window
    .partitionBy("user_id")
    .orderBy("event_ts")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

events_running = events_feat.withColumn("cumulative_events", F.count(F.lit(1)).over(w_user_running))
events_running.select("user_id","event_ts","event_type","cumulative_events").show(20, truncate=False)

In [0]:
# Window function 2: Ranking brands by purchases (ranking demo)
brand_purchases = (
    events_feat
    .filter(F.col("event_type") == "purchase")
    .groupBy("brand")
    .agg(F.count("*").alias("purchase_cnt"))
)

w_rank = Window.orderBy(F.desc("purchase_cnt"))

ranked_brands = brand_purchases.withColumn("brand_rank", F.dense_rank().over(w_rank))
ranked_brands.orderBy("brand_rank").show(20, truncate=False)


In [0]:
# Conversion rate by category (PDF practice, made safe)

cat_counts = (
    events_feat
    .groupBy("category_code", "event_type")
    .count()
    .groupBy("category_code")
    .pivot("event_type")
    .sum("count")
)

cat_conv = (
    cat_counts
    .withColumn("view", F.coalesce(F.col("view"), F.lit(0)))
    .withColumn("purchase", F.coalesce(F.col("purchase"), F.lit(0)))
    .withColumn(
        "conversion_rate",
        F.when(F.col("view") == 0, F.lit(0.0)).otherwise(F.col("purchase") / F.col("view") * 100)
    )
    .orderBy(F.desc("conversion_rate"))
)

cat_conv.select("category_code", "view", "purchase", "conversion_rate").show(20, truncate=False)


In [0]:
# UDF example

@F.udf(returnType=StringType())
def normalize_brand(b):
    if b is None:
        return None
    b = b.strip().lower()
    return b if b else None

events_udf = events_feat.withColumn("brand_norm", normalize_brand(F.col("brand")))
events_udf.select("brand","brand_norm").show(20, truncate=False)
